# Baseline Model: TF-IDF + Logistic Regression
## VioletPro Internship - Day 5-6

**Objective:** Build and evaluate baseline expense classifier  
**Model:** TF-IDF Vectorizer + Logistic Regression  
**Date:** November 12, 2026

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    classification_report,
    confusion_matrix
)
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Libraries imported successfully!')

## 2. Load Processed Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/sample_transactions.csv')

print(f'Dataset loaded: {df.shape[0]} transactions')
print(f'Categories: {df["category"].nunique()}')
print(f'\nCategory distribution:')
print(df['category'].value_counts())

## 3. Train/Validation/Test Split

In [ ]:
# Prepare features and labels
X = df['description']
y = df['category']

# Split: 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=RANDOM_STATE  # 0.176 * 0.85 ≈ 0.15
)

print(f'Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)')
print(f'Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)')
print(f'\nTotal: {len(X)} samples')

## 4. Build Baseline Model Pipeline

In [ ]:
# Create pipeline: TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=1000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        lowercase=True,
        strip_accents='unicode'
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver='lbfgs',
        multi_class='multinomial'
    ))
])

print('✅ Pipeline created:')
print(pipeline)

## 5. Train the Model

In [ ]:
print('Training baseline model...')

# Train
pipeline.fit(X_train, y_train)

print('✅ Model trained successfully!')

## 6. Evaluation

In [ ]:
# Predictions
y_train_pred = pipeline.predict(X_train)
y_val_pred = pipeline.predict(X_val)
y_test_pred = pipeline.predict(X_test)

# Calculate metrics
def evaluate_model(y_true, y_pred, set_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print(f'\n{set_name} Set Performance:')
    print(f'  Accuracy:  {acc:.4f} ({acc*100:.2f}%)')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    
    return acc, prec, rec, f1

# Evaluate on all sets
train_metrics = evaluate_model(y_train, y_train_pred, 'Training')
val_metrics = evaluate_model(y_val, y_val_pred, 'Validation')
test_metrics = evaluate_model(y_test, y_test_pred, 'Test')

### 6.1 Detailed Classification Report

In [ ]:
print('\nDetailed Classification Report (Test Set):\n')
print(classification_report(y_test, y_test_pred))

### 6.2 Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
categories = sorted(y.unique())

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=categories, yticklabels=categories)
plt.title('Confusion Matrix - Baseline Model', fontsize=16, fontweight='bold')
plt.ylabel('True Category', fontsize=12)
plt.xlabel('Predicted Category', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../models/confusion_matrix_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Confusion matrix saved')

### 6.3 Performance Comparison

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame({
    'Set': ['Training', 'Validation', 'Test'],
    'Accuracy': [train_metrics[0], val_metrics[0], test_metrics[0]],
    'Precision': [train_metrics[1], val_metrics[1], test_metrics[1]],
    'Recall': [train_metrics[2], val_metrics[2], test_metrics[2]],
    'F1-Score': [train_metrics[3], val_metrics[3], test_metrics[3]]
})

print('\nPerformance Summary:')
print(results_df.to_string(index=False))

# Plot comparison
results_df.set_index('Set')[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(
    kind='bar', figsize=(10, 6), rot=0
)
plt.title('Baseline Model Performance', fontsize=14, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../models/performance_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Feature Importance (Top TF-IDF Terms)

In [ ]:
# Get feature names and coefficients
feature_names = pipeline.named_steps['tfidf'].get_feature_names_out()
coefficients = pipeline.named_steps['clf'].coef_

# Show top features for each category
categories = pipeline.named_steps['clf'].classes_
n_top = 10

print('\nTop 10 Important Features per Category:\n')
for i, category in enumerate(categories):
    top_indices = np.argsort(coefficients[i])[-n_top:][::-1]
    top_features = [feature_names[idx] for idx in top_indices]
    top_scores = [coefficients[i][idx] for idx in top_indices]
    
    print(f'{category}:')
    for feat, score in zip(top_features, top_scores):
        print(f'  {feat}: {score:.4f}')
    print()

## 8. Test Predictions on Sample Data

In [ ]:
# Test on sample descriptions
test_samples = [
    "Uber ride to downtown",
    "Starbucks coffee",
    "Netflix subscription",
    "CVS Pharmacy medicine",
    "Electric bill payment",
    "Amazon purchase",
    "Rent payment",
    "Online course Udemy"
]

print('Sample Predictions:\n')
for sample in test_samples:
    pred = pipeline.predict([sample])[0]
    proba = pipeline.predict_proba([sample])[0]
    confidence = max(proba) * 100
    print(f'Description: "{sample}"')
    print(f'Predicted: {pred} (Confidence: {confidence:.1f}%)\n')

## 9. Save the Model

In [ ]:
# Save the trained pipeline
model_path = '../models/baseline_model.joblib'
joblib.dump(pipeline, model_path)

print(f'✅ Model saved to: {model_path}')

# Save metrics
metrics_df = pd.DataFrame({
    'Model': ['Baseline (TF-IDF + LogReg)'],
    'Train_Acc': [train_metrics[0]],
    'Val_Acc': [val_metrics[0]],
    'Test_Acc': [test_metrics[0]],
    'Test_F1': [test_metrics[3]]
})
metrics_df.to_csv('../models/baseline_metrics.csv', index=False)

print('✅ Metrics saved to: models/baseline_metrics.csv')

## 10. Summary & Next Steps

### Results Summary:
- **Model:** TF-IDF Vectorizer + Logistic Regression
- **Test Accuracy:** ~85-90% (check actual results above)
- **Key Strengths:** Fast training, interpretable, good baseline
- **Limitations:** May struggle with unseen words, no context understanding

### Next Steps (Days 7-9):
1. ✅ Baseline established
2. Try other models: Naive Bayes, Random Forest, SVM
3. Hyperparameter tuning with GridSearchCV
4. Model comparison and selection
5. Build prediction function and Streamlit demo